# Edge and Face Metrics with topologic_fast

This notebook demonstrates edge and face measurement operations using `topologic_fast`.

We'll cover:
- Edge properties: length, direction, angle, midpoint
- Face properties: area, perimeter, normal vector
- Finding longest/shortest edges and largest/smallest faces
- Visualization of geometric metrics

**Note**: This notebook is adapted from the topologicpy Edge_Face_Metrics tutorial.

## Import Libraries

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

print(f"topologic_fast version: {tf.__version__}")

## Visualization Helper Functions

In [ ]:
def get_mesh_data(cell, color='lightblue', opacity=0.5):
    """Convert a Cell to plotly Mesh3d data."""
    mesh = tf.Mesh.ByCell(cell)
    obj_content = mesh.ToOBJ()
    
    vertices = []
    faces = []
    
    for line in obj_content.strip().split('\n'):
        parts = line.strip().split()
        if not parts:
            continue
        if parts[0] == 'v':
            vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
        elif parts[0] == 'f':
            face_indices = [int(p.split('/')[0]) - 1 for p in parts[1:]]
            if len(face_indices) >= 3:
                faces.append(face_indices[:3])
    
    if not vertices or not faces:
        return None
    
    vertices = np.array(vertices)
    faces = np.array(faces)
    
    return go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        color=color,
        opacity=opacity,
        flatshading=True
    )


def draw_edge(fig, edge, color='black', width=2, name=None):
    """Add an edge to a plotly figure."""
    start = edge.StartVertex()
    end = edge.EndVertex()
    fig.add_trace(go.Scatter3d(
        x=[start.X(), end.X()],
        y=[start.Y(), end.Y()],
        z=[start.Z(), end.Z()],
        mode='lines',
        line=dict(color=color, width=width),
        name=name,
        showlegend=name is not None,
        hoverinfo='skip' if name is None else 'name'
    ))

## Create Sample Geometry

We'll create a compound shape by merging a prism and a cylinder to demonstrate edge and face metrics.

In [ ]:
# Create a prism (box)
prism = tf.Cell.Box(0, 0, 0, 2, 2, 5)
print(f"Prism: {len(prism.Faces())} faces, {len(prism.Edges())} edges")

# Create a cylinder
cylinder = tf.Cell.Cylinder(1, 1, 2.4, 2, 0.2, 24)
print(f"Cylinder: {len(cylinder.Faces())} faces, {len(cylinder.Edges())} edges")

# Merge them
merged = tf.Topology.Union(prism, cylinder)
print(f"Merged: {len(merged.Faces())} faces, {len(merged.Edges())} edges")

## Edge Metrics

Let's examine edge properties like length, direction, and midpoint.

In [ ]:
# Get all edges
edges = merged.Edges()
print(f"Total edges: {len(edges)}")

# Calculate edge lengths
edge_lengths = [(i, edge, edge.Length()) for i, edge in enumerate(edges)]

# Sort by length
edge_lengths.sort(key=lambda x: x[2], reverse=True)

print("\nEdge Length Statistics:")
print(f"  Longest edge: {edge_lengths[0][2]:.4f} (index {edge_lengths[0][0]})")
print(f"  Shortest edge: {edge_lengths[-1][2]:.4f} (index {edge_lengths[-1][0]})")
print(f"  Average length: {sum(e[2] for e in edge_lengths) / len(edge_lengths):.4f}")

In [ ]:
# Examine properties of a specific edge
sample_edge = edges[0]

print("Sample Edge Properties:")
print(f"  Length: {sample_edge.Length():.4f}")
print(f"  Direction: {sample_edge.Direction()}")
print(f"  Midpoint: {sample_edge.Midpoint()}")
print(f"  Start: ({sample_edge.StartVertex().X():.2f}, {sample_edge.StartVertex().Y():.2f}, {sample_edge.StartVertex().Z():.2f})")
print(f"  End: ({sample_edge.EndVertex().X():.2f}, {sample_edge.EndVertex().Y():.2f}, {sample_edge.EndVertex().Z():.2f})")

In [ ]:
# Calculate angles between edges
# Create two perpendicular edges for demonstration
edge_x = tf.Edge.ByCoordinates(0, 0, 0, 1, 0, 0)  # Along X axis
edge_y = tf.Edge.ByCoordinates(0, 0, 0, 0, 1, 0)  # Along Y axis
edge_z = tf.Edge.ByCoordinates(0, 0, 0, 0, 0, 1)  # Along Z axis
edge_diag = tf.Edge.ByCoordinates(0, 0, 0, 1, 1, 0)  # Diagonal in XY plane

print("Edge Angle Calculations (degrees):")
print(f"  X vs Y axis: {tf.Edge.Angle(edge_x, edge_y):.1f}")
print(f"  X vs Z axis: {tf.Edge.Angle(edge_x, edge_z):.1f}")
print(f"  X vs XY diagonal: {tf.Edge.Angle(edge_x, edge_diag):.1f}")

# Check collinearity and parallelism
edge_x2 = tf.Edge.ByCoordinates(2, 0, 0, 3, 0, 0)  # Parallel to X
print(f"\n  edge_x is parallel to edge_x2: {tf.Edge.IsParallel(edge_x, edge_x2)}")
print(f"  edge_x is collinear with edge_x2: {tf.Edge.IsCollinear(edge_x, edge_x2)}")

## Find Longest and Shortest Edges

In [ ]:
# Find longest and shortest edges
def find_longest_edges(edges, n=3):
    """Find the n longest edges."""
    sorted_edges = sorted(edges, key=lambda e: e.Length(), reverse=True)
    return sorted_edges[:n]

def find_shortest_edges(edges, n=3):
    """Find the n shortest edges."""
    sorted_edges = sorted(edges, key=lambda e: e.Length())
    return sorted_edges[:n]

longest = find_longest_edges(edges, 5)
shortest = find_shortest_edges(edges, 5)

print("Top 5 Longest Edges:")
for i, edge in enumerate(longest):
    print(f"  {i+1}. Length = {edge.Length():.4f}")

print("\nTop 5 Shortest Edges:")
for i, edge in enumerate(shortest):
    print(f"  {i+1}. Length = {edge.Length():.4f}")

## Face Metrics

Now let's examine face properties like area, perimeter, and normal vectors.

In [ ]:
# Get all faces
faces = merged.Faces()
print(f"Total faces: {len(faces)}")

# Calculate face areas
face_areas = [(i, face, face.Area()) for i, face in enumerate(faces)]

# Sort by area
face_areas.sort(key=lambda x: x[2], reverse=True)

print("\nFace Area Statistics:")
print(f"  Largest face: {face_areas[0][2]:.4f} (index {face_areas[0][0]})")
print(f"  Smallest face: {face_areas[-1][2]:.4f} (index {face_areas[-1][0]})")
print(f"  Average area: {sum(f[2] for f in face_areas) / len(face_areas):.4f}")
print(f"  Total surface area: {sum(f[2] for f in face_areas):.4f}")

In [ ]:
# Examine properties of sample faces
print("Face Properties Sample:")
print("=" * 60)

for i in range(min(5, len(faces))):
    face = faces[i]
    normal = face.Normal()
    center = face.CenterOfMass()
    
    print(f"\nFace {i}:")
    print(f"  Area: {face.Area():.4f}")
    print(f"  Perimeter: {face.Perimeter():.4f}")
    print(f"  Normal: ({normal[0]:.3f}, {normal[1]:.3f}, {normal[2]:.3f})")
    print(f"  Center: ({center[0]:.3f}, {center[1]:.3f}, {center[2]:.3f})")
    print(f"  Vertices: {len(face.Vertices())}")
    print(f"  Edges: {len(face.Edges())}")

## Find Largest and Smallest Faces

In [ ]:
def find_largest_faces(faces, n=3):
    """Find the n largest faces by area."""
    sorted_faces = sorted(faces, key=lambda f: f.Area(), reverse=True)
    return sorted_faces[:n]

def find_smallest_faces(faces, n=3):
    """Find the n smallest faces by area."""
    sorted_faces = sorted(faces, key=lambda f: f.Area())
    return sorted_faces[:n]

largest_faces = find_largest_faces(faces, 3)
smallest_faces = find_smallest_faces(faces, 3)

print("Top 3 Largest Faces:")
for i, face in enumerate(largest_faces):
    print(f"  {i+1}. Area = {face.Area():.4f}, Normal = {face.Normal()}")

print("\nTop 3 Smallest Faces:")
for i, face in enumerate(smallest_faces):
    print(f"  {i+1}. Area = {face.Area():.4f}, Normal = {face.Normal()}")

## Visualize Metrics

In [ ]:
# Visualize the geometry with highlighted extreme edges
fig = go.Figure()

# Add mesh
mesh_data = get_mesh_data(merged, color='lightgray', opacity=0.3)
if mesh_data:
    mesh_data.name = 'Geometry'
    fig.add_trace(mesh_data)

# Draw all edges in light gray
for edge in edges:
    draw_edge(fig, edge, color='lightgray', width=1)

# Highlight longest edges in red
for i, edge in enumerate(longest[:3]):
    draw_edge(fig, edge, color='red', width=6, 
              name=f'Longest {i+1} ({edge.Length():.2f})')

# Highlight shortest edges in blue
for i, edge in enumerate(shortest[:3]):
    draw_edge(fig, edge, color='blue', width=6,
              name=f'Shortest {i+1} ({edge.Length():.2f})')

fig.update_layout(
    title='Edge Metrics: Longest (red) vs Shortest (blue)',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=900,
    height=600
)

fig.show()

In [ ]:
# Visualize faces with area-based coloring
fig = go.Figure()

# Color faces by area
areas = [face.Area() for face in faces]
min_area, max_area = min(areas), max(areas)
area_range = max_area - min_area if max_area > min_area else 1

# Draw each face
for i, face in enumerate(faces):
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    if len(coords) >= 3:
        x = [c[0] for c in coords]
        y = [c[1] for c in coords]
        z = [c[2] for c in coords]
        
        # Color based on area (red = large, blue = small)
        normalized_area = (face.Area() - min_area) / area_range
        r = int(255 * normalized_area)
        b = int(255 * (1 - normalized_area))
        color = f'rgb({r}, 100, {b})'
        
        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            color=color,
            opacity=0.7,
            alphahull=0,
            name=f'Face {i} (A={face.Area():.2f})',
            showlegend=False,
            hovertext=f'Face {i}\nArea: {face.Area():.4f}',
            hoverinfo='text'
        ))

# Highlight largest faces
for face in largest_faces:
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    if len(coords) >= 3:
        # Draw outline
        for j in range(len(coords)):
            p1 = coords[j]
            p2 = coords[(j + 1) % len(coords)]
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='orange', width=6),
                showlegend=False
            ))

# Highlight smallest faces
for face in smallest_faces:
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    if len(coords) >= 3:
        for j in range(len(coords)):
            p1 = coords[j]
            p2 = coords[(j + 1) % len(coords)]
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='cyan', width=6),
                showlegend=False
            ))

fig.update_layout(
    title='Face Metrics: Color by Area (red=large, blue=small), Outlined: orange=largest, cyan=smallest',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=900,
    height=600
)

fig.show()

## Visualize Face Normals

In [ ]:
# Visualize face normals
fig = go.Figure()

# Add mesh
mesh_data = get_mesh_data(merged, color='lightblue', opacity=0.4)
if mesh_data:
    fig.add_trace(mesh_data)

# Draw normal vectors at face centers
normal_scale = 0.5  # Scale factor for normal display

for face in faces:
    center = face.CenterOfMass()
    normal = face.Normal()
    
    # Calculate normal endpoint
    end = (
        center[0] + normal[0] * normal_scale,
        center[1] + normal[1] * normal_scale,
        center[2] + normal[2] * normal_scale
    )
    
    # Draw normal vector
    fig.add_trace(go.Scatter3d(
        x=[center[0], end[0]],
        y=[center[1], end[1]],
        z=[center[2], end[2]],
        mode='lines',
        line=dict(color='red', width=3),
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Draw arrowhead (small sphere at tip)
    fig.add_trace(go.Scatter3d(
        x=[end[0]],
        y=[end[1]],
        z=[end[2]],
        mode='markers',
        marker=dict(size=4, color='red'),
        showlegend=False,
        hovertext=f'Normal: ({normal[0]:.2f}, {normal[1]:.2f}, {normal[2]:.2f})',
        hoverinfo='text'
    ))

fig.update_layout(
    title='Face Normal Vectors',
    scene=dict(
        aspectmode='data',
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=800,
    height=600
)

fig.show()

## Additional Edge Operations

In [ ]:
# Demonstrate various edge operations
test_edge = tf.Edge.ByCoordinates(0, 0, 0, 10, 0, 0)

print("Edge Operations Demo:")
print("=" * 50)
print(f"Original edge: (0,0,0) to (10,0,0)")
print(f"  Length: {test_edge.Length()}")

# Normalize
normalized = test_edge.Normalize()
print(f"\nNormalized edge length: {normalized.Length():.4f}")

# Set length
scaled = test_edge.SetLength(5.0)
print(f"Set to length 5: {scaled.Length():.4f}")

# Reverse
reversed_edge = test_edge.Reverse()
print(f"\nReversed edge:")
print(f"  Start: ({reversed_edge.StartVertex().X()}, {reversed_edge.StartVertex().Y()}, {reversed_edge.StartVertex().Z()})")
print(f"  End: ({reversed_edge.EndVertex().X()}, {reversed_edge.EndVertex().Y()}, {reversed_edge.EndVertex().Z()})")

# Trim
trimmed = test_edge.Trim(0.2, 0.8)  # Keep 20% to 80%
print(f"\nTrimmed (20%-80%):")
print(f"  Length: {trimmed.Length():.4f} (expected: 6.0)")

# Split
e1, e2 = test_edge.SplitAt(0.3)  # Split at 30%
print(f"\nSplit at 30%:")
print(f"  Part 1 length: {e1.Length():.4f}")
print(f"  Part 2 length: {e2.Length():.4f}")

# Vertex by parameter
v_mid = test_edge.VertexByParameter(0.5)
print(f"\nVertex at parameter 0.5: ({v_mid.X()}, {v_mid.Y()}, {v_mid.Z()})")

# Vertex by distance
v_dist = test_edge.VertexByDistance(3.0)
print(f"Vertex at distance 3.0: ({v_dist.X()}, {v_dist.Y()}, {v_dist.Z()})")

## Creating Faces

In [ ]:
# Create various faces and examine their properties
print("Face Creation Demo:")
print("=" * 50)

# Rectangle
rect = tf.Face.Rectangle(width=5, length=3)
print(f"\nRectangle (5x3):")
print(f"  Area: {rect.Area():.4f} (expected: 15)")
print(f"  Perimeter: {rect.Perimeter():.4f} (expected: 16)")
print(f"  Normal: {rect.Normal()}")

# Circle
circle = tf.Face.Circle(radius=2.0, sides=64)
expected_area = np.pi * 2**2
print(f"\nCircle (r=2):")
print(f"  Area: {circle.Area():.4f} (expected: {expected_area:.4f})")
print(f"  Perimeter: {circle.Perimeter():.4f} (expected: {2 * np.pi * 2:.4f})")

# Face with hole
outer = tf.Wire.Rectangle(width=6, length=6)
inner = tf.Wire.Rectangle(width=2, length=2)
face_with_hole = tf.Face.ByExternalInternalBoundaries(outer, [inner])
print(f"\nFace with hole:")
print(f"  Area: {face_with_hole.Area():.4f} (expected: 36 - 4 = 32)")

## Note on topologic_fast vs topologicpy

Some topologicpy methods may not be directly available in topologic_fast:

- `Topology.LongestEdges()` - Find longest edges (implemented manually above)
- `Topology.ShortestEdges()` - Find shortest edges (implemented manually above)
- `Topology.LargestFaces()` - Find largest faces (implemented manually above)
- `Topology.SmallestFaces()` - Find smallest faces (implemented manually above)
- `Topology.RemoveCoplanarFaces()` - Remove coplanar faces
- Dictionary-based styling for visualization

The core metrics (Length, Area, Normal, etc.) are available and demonstrated above.

## Summary

This notebook demonstrated:

### Edge Metrics
- `edge.Length()` - Get edge length
- `edge.Direction()` - Get direction vector
- `edge.Midpoint()` - Get midpoint coordinates
- `tf.Edge.Angle(e1, e2)` - Angle between edges
- `tf.Edge.IsParallel(e1, e2)` - Check parallelism
- `tf.Edge.IsCollinear(e1, e2)` - Check collinearity

### Edge Operations
- `edge.Normalize()` - Create unit length edge
- `edge.SetLength(length)` - Set edge length
- `edge.Reverse()` - Reverse direction
- `edge.Trim(start, end)` - Trim edge
- `edge.SplitAt(param)` - Split edge
- `edge.VertexByParameter(t)` - Get point on edge

### Face Metrics
- `face.Area()` - Get face area
- `face.Perimeter()` - Get face perimeter
- `face.Normal()` - Get normal vector
- `face.CenterOfMass()` - Get centroid

### Applications
- Quality analysis of 3D models
- Finding extreme geometric features
- Mesh analysis and optimization
- Architectural/engineering analysis